# Outcome Engine batch template
Install a reviewed 0.5.0 wheel and requirements-dbt.txt in the Fabric Environment, attach a Lakehouse, and mark the next cell as the parameter cell in Fabric. This template is not a deployed service. Each run uses local temporary database files and publishes immutable exports; it does not provide a central reviewer database or browser execution.


In [ ]:
suppliers_path = "/lakehouse/default/Files/outcome/input/suppliers.csv"
spend_path = "/lakehouse/default/Files/outcome/input/spend.csv"
output_dir = "/lakehouse/default/Files/outcome/runs"
pair_model_path = None  # Optional reviewed supplier artifact; never the SPIDER model
matching_config_path = None  # Exact configuration used to train that artifact
dataset_namespace = "client_a"  # Stable client/source namespace, not an authorization boundary


In [ ]:
import json
from pathlib import Path
import shutil
from tempfile import TemporaryDirectory
from enterprise_ai.pipeline import run_pipeline
from enterprise_ai.matching import MatchConfig
pair_model = json.loads(Path(pair_model_path).read_text()) if pair_model_path else None
config = MatchConfig(**json.loads(Path(matching_config_path).read_text())) if matching_config_path else MatchConfig()

with TemporaryDirectory(prefix="outcome-fabric-") as local:
    manifest = run_pipeline(suppliers_path, spend_path, Path(local)/"runs", config=config, pair_model=pair_model, dataset_namespace=dataset_namespace)
    source = Path(manifest["output_dir"])
    target = Path(output_dir)/source.name
    target.mkdir(parents=True, exist_ok=False)
    for name in ("state.json", "resolution-audit.jsonl", "report.md"):
        shutil.copyfile(source/name, target/name)
    for relative in ("contracts/dbt.log", "contracts/project/target/run_results.json"):
        output = target/relative
        output.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source/relative, output)
    manifest["output_dir"] = str(target)
    # Completion marker last; downstream activities ignore directories without it.
    (target/"manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
